# Baseline Linear Regression

Time-based train/test split, baseline numeric features, median imputation, plain LinearRegression.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
pd.set_option('display.max_columns', 60)

**Why MAE?** MAE is directly interpretable in seconds — a MAE of 600 s means we're off by ~10 min on average. RMSE is reported alongside it for sensitivity to large errors. R² contextualises how much variance the model explains. Accuracy is not applicable (continuous target).

## 1. Load, build target, time-based split

In [2]:
df = pd.read_csv('../data/historical_data.csv', parse_dates=['created_at', 'actual_delivery_time'])
df['delivery_duration'] = (df['actual_delivery_time'] - df['created_at']).dt.total_seconds()

# Drop missing target + obvious outliers (60s .. 3h)
df = df.dropna(subset=['delivery_duration'])
df = df[(df['delivery_duration'] >= 60) & (df['delivery_duration'] <= 3*3600)]

# Sort by time and 80/20 split
df = df.sort_values('created_at').reset_index(drop=True)
cutoff = int(len(df) * 0.8)
train, test = df.iloc[:cutoff].copy(), df.iloc[cutoff:].copy()
print('train:', train.shape, ' test:', test.shape)
print('train date range:', train['created_at'].min(), '→', train['created_at'].max())
print('test  date range:', test['created_at'].min(),  '→', test['created_at'].max())

train: (157826, 17)  test: (39457, 17)
train date range: 2015-01-21 15:22:03 → 2015-02-13 04:06:37
test  date range: 2015-02-13 04:06:44 → 2015-02-18 06:00:44


## 2. Select baseline features

In [3]:
BASELINE_FEATURES = [
    'total_items', 'subtotal', 'num_distinct_items',
    'min_item_price', 'max_item_price',
    'estimated_order_place_duration',
    'estimated_store_to_consumer_driving_duration',
]

## 3. Median imputation (computed on train only)

In [4]:
medians = train[BASELINE_FEATURES].median()
Xb_train = train[BASELINE_FEATURES].fillna(medians)
Xb_test  = test[BASELINE_FEATURES].fillna(medians)
y_train = train['delivery_duration'].values
y_test  = test['delivery_duration'].values
medians.to_frame('median').T

,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,estimated_order_place_duration,estimated_store_to_consumer_driving_duration
median,3.0,2200.0,2.0,595.0,1095.0,251.0,543.0


In [5]:
# Majority-class analogue for regression: predict the train mean for every row.
# Any model must beat this floor to be useful.
mean_pred = np.full(len(y_test), y_train.mean())
print(f'Mean-prediction floor   MAE = {mean_absolute_error(y_test, mean_pred):.1f} s')

Mean-prediction floor   MAE = 834.8 s


## 4. Train Linear Regression

In [6]:
def regression_metrics(y_true, y_pred):
    return {
        'MAE':  mean_absolute_error(y_true, y_pred),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'R2':   r2_score(y_true, y_pred),
    }

lr_baseline = LinearRegression().fit(Xb_train, y_train)
preds_baseline = lr_baseline.predict(Xb_test)
metrics_baseline = regression_metrics(y_test, preds_baseline)
pd.DataFrame({'Linear Regression (baseline)': metrics_baseline}).T.round(2)

,MAE,RMSE,R2
Linear Regression (baseline),779.83,1061.75,0.09


## 5. Coefficient Interpretation

In [7]:
pd.Series(lr_baseline.coef_, index=Xb_train.columns).sort_values().rename('coef').to_frame()

,coef
min_item_price,0.009243
max_item_price,0.067372
subtotal,0.091751
total_items,0.984761
estimated_order_place_duration,1.151830
estimated_store_to_consumer_driving_duration,1.174950
num_distinct_items,30.280331


In [8]:
import os; os.makedirs('../reports', exist_ok=True)
train.to_parquet('../reports/train.parquet') if False else train.to_csv('../reports/train.csv', index=False)
test.to_csv('../reports/test.csv', index=False)
pd.DataFrame({'Linear Regression (baseline)': metrics_baseline}).T.to_csv('../reports/metrics_baseline.csv')
print('saved.')

saved.
